In [6]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path=Path.cwd()/"pass.env"
loaded=load_dotenv(dotenv_path=env_path,override=True)

print(f"¿Archivo encontrado y cargado?: {loaded}")

PG_CONNECTION_STRING=os.getenv("PG_CONNECTION_STRING")

if not PG_CONNECTION_STRING:
    raise ValueError("Falta PG_CONNECTION_STRING en pass.env")

print("Variables cargadas correctamente.")


¿Archivo encontrado y cargado?: True
Variables cargadas correctamente.


In [7]:
import psycopg2

conn=psycopg2.connect(PG_CONNECTION_STRING)
cur=conn.cursor()

cur.execute("SELECT current_database(),current_user;")
bd,usuario=cur.fetchone()

print("Conectado a PostgreSQL")
print("Base de datos:",bd)
print("Usuario:",usuario)


Conectado a PostgreSQL
Base de datos: neondb
Usuario: neondb_owner


In [8]:
# Verificar que la tabla roles ya exista.
# El esquema nuevo también inserta estos roles; este notebook se conserva
# como herramienta idempotente para asegurar que estén presentes y activos.

cur.execute("""
SELECT EXISTS(
    SELECT 1
    FROM information_schema.tables
    WHERE table_schema='public'
      AND table_name='roles'
);
""")

if not cur.fetchone()[0]:
    raise RuntimeError("La tabla roles no existe. Ejecute primero creacion_tablas.ipynb o squema_bd.sql.")

roles=[
    ('Admin','Administración general, gestión de usuarios y restauración de registros'),
    ('Medico','Consulta y gestión de la información clínica autorizada'),
    ('Administrativo','Gestión exclusiva de facturación y datos administrativos'),
    ('Paciente','Consulta de información propia y creación de reportes previos')
]

try:
    for nombre,descripcion in roles:
        cur.execute("""
        INSERT INTO roles(nombre,descripcion,is_active)
        VALUES(%s,%s,TRUE)
        ON CONFLICT(nombre)
        DO UPDATE SET
            descripcion=EXCLUDED.descripcion,
            is_active=TRUE;
        """,(nombre,descripcion))

    conn.commit()
    print("Roles creados/actualizados correctamente.")
except Exception:
    conn.rollback()
    print("Ocurrió un error. Se hizo rollback.")
    raise


Roles creados/actualizados correctamente.


In [9]:
# Verificar el resultado final
cur.execute("""
SELECT id_rol,nombre,descripcion,is_active
FROM roles
ORDER BY id_rol;
""")

roles_bd=cur.fetchall()

for rol in roles_bd:
    print(rol)

nombres={rol[1] for rol in roles_bd if rol[3]}
esperados={'Admin','Medico','Administrativo','Paciente'}

if nombres!=esperados:
    raise RuntimeError(f"Los roles activos no coinciden con los esperados. Actuales: {sorted(nombres)}")

print("Los 4 roles esperados están activos.")


(1, 'Admin', 'Administración general, gestión de usuarios y restauración de registros', True)
(2, 'Medico', 'Consulta y gestión de la información clínica autorizada', True)
(3, 'Administrativo', 'Gestión exclusiva de facturación y datos administrativos', True)
(4, 'Paciente', 'Consulta de información propia y creación de reportes previos', True)
Los 4 roles esperados están activos.


In [10]:
cur.close()
conn.close()
print("Conexión cerrada correctamente.")


Conexión cerrada correctamente.
